# 최적화된 GPTQ 양자화 (베이스라인 기반 개선)

## 조사 결과 기반 최적화

### 참고 자료
- [vLLM Quantization Guide](https://docs.vllm.ai/en/latest/features/quantization/)
- [LLM Compressor GPTQModifier](https://docs.vllm.ai/projects/llm-compressor/en/stable/reference/llmcompressor/modifiers/quantization/gptq/)
- [vLLM Quantization Benchmarks](https://docs.jarvislabs.ai/blog/vllm-quantization-complete-guide-benchmarks)

### 베이스라인 대비 변경점

| 항목 | 베이스라인 | 최적화 버전 | 효과 |
|------|-----------|------------|------|
| actorder | (미지정) | **weight** | 정확도 ~2점 향상 |
| block_size | (미지정) | **128** | Marlin 커널 호환 |
| dampening_frac | (미지정) | **0.001** | Hessian 안정화 |
| symmetric | (미지정) | **true** | 추론 속도 향상 |

### 핵심 발견
1. **actorder="weight"**: W4A16에서 정확도 최대 2점 향상 (llmcompressor v0.8.0+)
2. **Marlin 커널**: vLLM이 자동 적용, GPTQ 2.6배 속도 향상
3. **group_size=128**: Marlin 호환 필수 조건

---

# 1. Import

In [1]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ CPU 모드로 실행됩니다")
print("\n✅ Import 완료!")

PyTorch: 2.9.1
CUDA: False
⚠️ CPU 모드로 실행됩니다

✅ Import 완료!


# 2. 설정 (베이스라인 + 최적화)

In [2]:
# ============================================================================
# 모델 설정 (베이스라인 동일)
# ============================================================================
MODEL_ID = "./open/base_model"  # 로컬 모델
OUT_DIR = "./model"             # 제출용 폴더명

# ============================================================================
# 데이터셋 설정 (베이스라인 동일)
# ============================================================================
DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

# ============================================================================
# 캘리브레이션 설정 (베이스라인 동일)
# ============================================================================
NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 512

# ============================================================================
# 양자화 설정 (베이스라인 동일)
# ============================================================================
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE = ["embed_tokens", "lm_head"]

# ============================================================================
# ⭐ 최적화 설정 (NEW!)
# ============================================================================
# 참고: https://docs.vllm.ai/projects/llm-compressor/en/stable/reference/llmcompressor/modifiers/quantization/gptq/

BLOCK_SIZE = 128        # Marlin 커널 호환 (필수)
DAMPENING_FRAC = 0.001  # Hessian 안정화 (기본값)
ACTORDER = "weight"     # W4A16 정확도 향상 (v0.8.0+ 권장)

# 원본 모델 크기
ORIGINAL_MODEL_SIZE_GB = 2.56

print("=" * 60)
print("최적화된 GPTQ 설정")
print("=" * 60)
print(f"MODEL_ID: {MODEL_ID}")
print(f"SCHEME: {SCHEME}")
print(f"SAMPLES: {NUM_CALIBRATION_SAMPLES}")
print(f"MAX_LEN: {MAX_SEQUENCE_LENGTH}")
print("---")
print("⭐ 최적화 파라미터:")
print(f"  BLOCK_SIZE: {BLOCK_SIZE} (Marlin 호환)")
print(f"  DAMPENING_FRAC: {DAMPENING_FRAC}")
print(f"  ACTORDER: {ACTORDER} (정확도 향상)")
print("=" * 60)

최적화된 GPTQ 설정
MODEL_ID: ./open/base_model
SCHEME: W4A16
SAMPLES: 256
MAX_LEN: 512
---
⭐ 최적화 파라미터:
  BLOCK_SIZE: 128 (Marlin 호환)
  DAMPENING_FRAC: 0.001
  ACTORDER: weight (정확도 향상)


# 3. 모델 로드

In [3]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

# GPU/CPU 자동 선택
if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32,
        trust_remote_code=True,
        device_map="cpu",
    )

print(f"[INFO] 모델 파라미터: {model.num_parameters():,}")
print("[INFO] 모델/토크나이저 로드 완료")

`torch_dtype` is deprecated! Use `dtype` instead!


[INFO] 모델 로드 중...
[INFO] 모델 파라미터: 1,279,391,488
[INFO] 모델/토크나이저 로드 완료


# 4. 데이터셋 로드

In [4]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)

print(f"[INFO] 데이터셋 크기: {len(ds)}")
print("[INFO] 데이터 전처리 완료")

[INFO] 캘리브레이션 데이터 로드 중...
[INFO] 데이터셋 크기: 256
[INFO] 데이터 전처리 완료


# 5. GPTQ 양자화 (최적화 버전)

### 베이스라인 vs 최적화

```python
# 베이스라인
GPTQModifier(
    scheme=SCHEME,
    targets=TARGETS,
    ignore=IGNORE,
)

# 최적화 (아래)
GPTQModifier(
    scheme=SCHEME,
    targets=TARGETS,
    ignore=IGNORE,
    block_size=128,       # ⭐ Marlin 호환
    dampening_frac=0.001, # ⭐ Hessian 안정화
    actorder="weight",    # ⭐ 정확도 향상
)
```

In [5]:
print("[INFO] GPTQ 양자화 시작 (최적화 버전)")
print(f"  - scheme: {SCHEME}")
print(f"  - samples: {NUM_CALIBRATION_SAMPLES}")
print(f"  - max_len: {MAX_SEQUENCE_LENGTH}")
print(f"  - block_size: {BLOCK_SIZE} (Marlin 호환)")
print(f"  - actorder: {ACTORDER} (정확도 향상)")
print(f"  - dampening_frac: {DAMPENING_FRAC}")

if torch.cuda.is_available():
    print("\n🚀 GPU 모드: 10-20분 예상\n")
else:
    print("\n⏳ CPU 모드: 15-30분 예상\n")

# ============================================================================
# 최적화된 GPTQ 레시피
# ============================================================================
recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
        # ⭐ 최적화 파라미터
        block_size=BLOCK_SIZE,        # Marlin 커널 호환
        dampening_frac=DAMPENING_FRAC, # Hessian 안정화
        actorder=ACTORDER,            # 가중치 기반 정렬 (정확도↑)
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("\n[INFO] GPTQ 양자화 완료!")

[INFO] GPTQ 양자화 시작 (최적화 버전)
  - scheme: W4A16
  - samples: 256
  - max_len: 512
  - block_size: 128 (Marlin 호환)
  - actorder: weight (정확도 향상)
  - dampening_frac: 0.001

⏳ CPU 모드: 15-30분 예상



Tokenizing:   0%|          | 0/256 [00:00<?, ? examples/s]

2026-02-10T14:31:23.547322+0900 | reset | INFO - Compression lifecycle reset
2026-02-10T14:31:23.549407+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-10T14:31:23.578409+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-10T14:31:23.579022+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`
2026-02-10T14:31:23.586487+0900 | dispatch_for_sequential | WARNING - CUDA/XPU is not available! Compressing model on CPU instead


W0210 14:31:23.619000 77652 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(1/31): Calibrating: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [01:05<00:00,  3.91it/s]

2026-02-10T14:32:29.366854+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 256 samples


...

2026-02-10T15:09:54.240882+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.q_proj using 256 samples


2026-02-10T15:09:54.666487+0900 | compress | METRIC - time 0.43s
2026-02-10T15:09:54.666978+0900 | compress | METRIC - error 3182.92
2026-02-10T15:09:54.668476+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-10T15:09:54.668833+0900 | compress | METRIC - Compressed module size: 16.941056 MB
2026-02-10T15:09:54.670752+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 256 samples
2026-02-10T15:09:54.938559+0900 | compress | METRIC - time 0.27s
2026-02-10T15:09:54.939351+0900 | compress | METRIC - error 902.42
2026-02-10T15:09:54.940503+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-10T15:09:54.940844+0900 | compress | METRIC - Compressed module size: 4.235264 MB
2026-02-10T15:09:54.941778+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 256 samples
2026-02-10T15:09:55.205383+0900 | compress | METRIC - time 0.26s
2026-02-10T15:0

(31/31): Propagating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:00<00:00, 3423.77it/s]

2026-02-10T15:10:09.784815+0900 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-10T15:10:09.792343+0900 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`



[INFO] GPTQ 양자화 완료!


# 6. 모델 저장

In [6]:
print("[INFO] 모델 저장 중...")

# 기존 폴더 삭제
if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

# 저장된 파일 확인
print(f"\n[INFO] 저장된 파일:")
total_size = 0
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    total_size += size
    print(f"  {f}: {size/1e6:.1f} MB")

quantized_size_gb = total_size / 1e9

print("\n" + "=" * 60)
print("모델 크기 비교")
print("=" * 60)
print(f"  원본 모델:     {ORIGINAL_MODEL_SIZE_GB:.2f} GB")
print(f"  양자화 모델:   {quantized_size_gb:.2f} GB")
print(f"  압축률:        {quantized_size_gb / ORIGINAL_MODEL_SIZE_GB * 100:.1f}%")
print("=" * 60)

[INFO] 모델 저장 중...
2026-02-10T15:10:09.835676+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:03, 61.68it/s]



[INFO] 저장된 파일:
  chat_template.jinja: 0.0 MB
  config.json: 0.0 MB
  generation_config.json: 0.0 MB
  merges.txt: 1.2 MB
  model.safetensors: 1407.7 MB
  recipe.yaml: 0.0 MB
  special_tokens_map.json: 0.0 MB
  tokenizer.json: 7.9 MB
  tokenizer_config.json: 0.1 MB
  vocab.json: 1.9 MB

모델 크기 비교
  원본 모델:     2.56 GB
  양자화 모델:   1.42 GB
  압축률:        55.4%


# 7. 제출 파일 생성

In [7]:
zip_name = "submit_optimized"
print(f"[INFO] {zip_name}.zip 생성 중...")

# 기존 zip 삭제
if os.path.exists(f"{zip_name}.zip"):
    os.remove(f"{zip_name}.zip")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

zip_size = os.path.getsize(f"{zip_name}.zip") / 1e9
print(f"[INFO] 생성 완료: {zip_name}.zip ({zip_size:.2f} GB)")

if zip_size <= 10:
    print("✅ 용량 제한 충족 (≤ 10GB)")
else:
    print("❌ 용량 초과!")

# 제출 구조 확인
print("\n" + "=" * 60)
print("제출 파일 구조")
print("=" * 60)
print(f"{zip_name}.zip")
print(f"└── model/")
for f in sorted(os.listdir(OUT_DIR))[:5]:
    print(f"    ├── {f}")
print("    └── ...")
print("=" * 60)

[INFO] submit_optimized.zip 생성 중...
[INFO] 생성 완료: submit_optimized.zip (0.88 GB)
✅ 용량 제한 충족 (≤ 10GB)

제출 파일 구조
submit_optimized.zip
└── model/
    ├── chat_template.jinja
    ├── config.json
    ├── generation_config.json
    ├── merges.txt
    ├── model.safetensors
    └── ...


---

# 예상 성능 개선

## 평가 공식
```
Score = 0.5 × PerfNorm + 0.5 × SpeedNorm
```

## 최적화 효과 분석

| 항목 | 베이스라인 | 최적화 버전 | 변화 |
|------|-----------|------------|------|
| **PerfNorm** | ~0.95 | ~0.97 | ↑ actorder="weight" |
| **SpeedNorm** | ~0.30 | ~0.30 | = (동일 압축률) |
| **Score** | ~0.625 | **~0.635** | ↑ |

## 추가 개선 방안

### 1. 캘리브레이션 품질 향상
```python
NUM_CALIBRATION_SAMPLES = 512  # 256 → 512
MAX_SEQUENCE_LENGTH = 1024     # 512 → 1024
```

### 2. AWQ 시도 (Marlin-AWQ가 가장 빠름)
- vLLM 벤치마크: AWQ 741 tok/s vs GPTQ 712 tok/s

---

## 참고 자료
- [vLLM Quantization Benchmarks](https://docs.jarvislabs.ai/blog/vllm-quantization-complete-guide-benchmarks)
- [LLM Compressor Docs](https://docs.vllm.ai/projects/llm-compressor/en/stable/)
- [GPTQModel GitHub](https://github.com/ModelCloud/GPTQModel)

---